# 12 — Clean ablation (EMS comment 6) + defensible 2040 scenario

All variants trained **in one run, same data, same windows, same seeds**. Each step changes ONE component:

| Model | Head | Rain (7) | GCN | Zone branches | Physics loss | Isolates |
|---|---|---|---|---|---|---|
| M1 | ✓ | | | | | baseline LSTM |
| M2 | ✓ | ✓ | | | | rain (M2 vs M1) |
| M2big | ✓ | ✓ | | | | capacity: hidden 112 ≈ M3.5 params (M3.5 vs M2big = zoning at equal size) |
| M3 | ✓ | ✓ | ✓ | | | GCN, no zones (M3 vs M2) |
| M3.5 | ✓ | ✓ | | ✓ | | zones (M3.5 vs M2) |
| M3.5G | ✓ | ✓ | ✓ | ✓ | | GCN inside zone model (M3.5G vs M3.5) |
| M4 | ✓ | ✓ | ✓ | ✓ | ✓ | physics only (M4 vs M3.5G) |

**Leakage fixes vs. earlier runs**
1. Early stopping on a **validation period inside training years (2018–2019)**. Test set (2020–2025) is never seen during training or model selection.
2. Rainfall features are **trailing (causal)** windows; climatology and normalisation fitted on ≤2019 only; built on the continuous block rainfall series (not the gappy well index).
3. Node features (mean/std/trend of head) from ≤2017 only.
4. Physics weight λ chosen on validation (mean over all seeds), not test.
5. Same wells, same window loop, same test samples as `04_baseline_benchmarks.py` → direct paired comparison with the baselines.

**Physics loss (M4) — the Methods section must describe exactly this:**
- Δĥᵢ(t) = ĥᵢ(t) − hᵢ(t−1) in metres
- *Spatial coherence*: L_sp = Σ wᵢⱼ (Δĥᵢ − Δĥⱼ)² / Σ wᵢⱼ over well pairs at the same month, wᵢⱼ = hydraulic conductance C = T_harm / L (T_eff thickness-weighted: Weathered 30, Massive 5, Fractured 100 m² d⁻¹)
- *Storage bound*: L_st = mean[ReLU(Δĥ − Δh_max,z)² + ReLU(Δh_min,z − Δĥ)²], bounds = 1st/99th percentile of observed monthly Δh in zone z (training years)
- L = MSE + λ (L_sp + L_st)

**Part B** (optional): free-run hindcast 2020–2025 vs. trend/climatology benchmarks, then refit on the full record (preprocessing fixed to training-period statistics) and project to 2040 under climatological rainfall, with a reversion-to-training-mean diagnostic.

Runs on Windows Jupyter (CPU) or Colab. No torch-geometric needed. CPU time ≈ 1.5–2 h for Part A, ≈ 45 min for Part B.

In [ ]:
# ============================== CELL 1 — CONFIG ==============================
import os, re, json, time, copy, warnings, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")

# ---- PATHS (edit only these) ----
BOOK2         = r"J:\Ground_water\Indore_gw\Book2.xlsx"
RAIN_DIR      = r"J:\NDCQ-2026-03-471\datafiles"
PIPELINE_FILE = r"J:\Indore_gw\01_indore_main_pipeline.py"   # GitHub copy used if missing
BASELINE_CSV  = r"J:\Indore_gw\baselines\baseline_perwell_metrics.csv"   # from 04
OUT_DIR       = r"J:\Indore_gw\clean_run"

# ---- SETUP ----
SEQ_LEN      = 24
HORIZON_TRIM = 12      # same window loop as 04 (last 12 months not used as targets)
TRAIN_END    = 2017    # train targets <= 2017
VAL_END      = 2019    # validation targets 2018-2019 ; test targets >= 2020
SEEDS        = [0, 1, 2]          # use [0,1,2,3,4] if time allows
MODELS          = ["M1", "M2", "M2big", "M3", "M3.5", "M3.5G", "M4"]
FORECAST_MODELS = ["M1", "M2", "M3", "M3.5", "M4"]   # Part B (hindcast + 2040)

HIDDEN, LAYERS, DROPOUT, GCN_DIM = 64, 2, 0.2, 16
LR, WEIGHT_DECAY = 1e-3, 1e-5
MAX_EPOCHS, PATIENCE = 300, 30
DATES_PER_BATCH = 4               # a batch = all wells at 4 months (needed for spatial term)
LAMBDA_GRID = [0.001, 0.01, 0.1]  # M4 physics weight, chosen on mean validation RMSE over SEEDS
GRAPH_RADIUS_KM = 15
T_EFF = {"Weathered": 30.0, "Massive": 5.0, "Fractured": 100.0}   # m2/day
ZONES = ["Weathered", "Massive", "Fractured"]

RUN_PART_B   = True               # hindcast + 2040 forecast
FORECAST_END = "2040-12-01"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)
torch.set_num_threads(max(1, os.cpu_count() - 1))
print("Device:", DEVICE, "| torch", torch.__version__, "| python", platform.python_version())

In [ ]:
# ===================== CELL 2 — LOAD DATA (same blocks as 04) =================
_needed = ["monthly_v3", "well_list_v3", "aq_info", "wells", "block_rain"]

def _block(lines, start, end, include_end=False):
    s = next(i for i, l in enumerate(lines) if start in l)
    e = next(i for i in range(s + 1, len(lines)) if end in lines[i])
    return "\n".join(lines[s:e + 1 if include_end else e])

if any(n not in globals() for n in _needed):
    if os.path.exists(PIPELINE_FILE):
        _src = open(PIPELINE_FILE, encoding="utf-8").read().splitlines()
    else:
        import urllib.request
        _url = ("https://raw.githubusercontent.com/Deepak1232095/"
                "pgnn-lstm-deccan-basalt/main/01_indore_main_pipeline.py")
        print("Pipeline file not found -> GitHub copy")
        _src = urllib.request.urlopen(_url).read().decode("utf-8").splitlines()
    _A = _block(_src, "DATA LOADING + BGL->MSL CONVERSION", "4. GRAPH CONSTRUCTION")
    _B = _block(_src, "EXCLUDE_WELLS = [", "well_list_v3 = sorted", include_end=True)
    _C = _block(_src, "COMPLETE RAINFALL INTEGRATION", "# Test on one well")
    _A = _A.replace("import shap", "")
    _A = re.sub(r"file_path\s*=\s*r?['\"].*?['\"]", lambda m: f"file_path = r'{BOOK2}'", _A)
    _A = re.sub(r"SAVE_DIR\s*=\s*r?['\"].*?['\"]", lambda m: f"SAVE_DIR = r'{OUT_DIR}'", _A)
    _C = re.sub(r"DATA_DIR\s*=\s*r?['\"].*?['\"]", lambda m: f"DATA_DIR = r'{RAIN_DIR}'", _C)
    _C = re.sub(r"SAVE_DIR\s*=\s*r?['\"].*?['\"]", lambda m: f"SAVE_DIR = r'{OUT_DIR}'", _C)
    for _n, _c in [("A data+zones", _A), ("B well filter", _B), ("C rainfall", _C)]:
        print("running block", _n)
        exec(compile(_c, f"<01:{_n}>", "exec"), globals())

print("\nWells in monthly_v3:", len(well_list_v3))
print("Rain blocks:", list(block_rain))

In [ ]:
# ============ CELL 3 — CAUSAL RAINFALL FEATURES (per block, continuous) =======
RAIN_NAMES = ["raw", "cum3", "cum6", "anom", "lag1", "lag2", "lag3"]

def _block_series(block):
    if block in block_rain:
        return block_rain[block].astype(float)
    return pd.concat(list(block_rain.values()), axis=1).mean(axis=1)

def rain_features(ts, norm_until=VAL_END):
    # 7 trailing features on a continuous monthly series; stats from <= norm_until
    ts = ts.asfreq("MS")
    r1 = ts.values.astype(float)
    trm = ts.index.year <= norm_until
    clim = ts[trm].groupby(ts.index[trm].month).mean()
    r2 = pd.Series(r1).rolling(3, min_periods=1).sum().values
    r3 = pd.Series(r1).rolling(6, min_periods=1).sum().values
    r4 = r1 - np.asarray(ts.index.month.map(clim), float)
    lags = [pd.Series(r1).shift(k).values.copy() for k in (1, 2, 3)]
    for k, l in enumerate(lags, 1):          # first k months: use climatology
        l[:k] = np.asarray(ts.index[:k].month.map(clim), float)
    X = np.column_stack([r1, r2, r3, r4] + lags)
    mu, sd = X[trm].mean(0), X[trm].std(0) + 1e-8
    return pd.DataFrame((X - mu) / sd, index=ts.index, columns=RAIN_NAMES)

def extend_with_climatology(ts, until, clim_from_year_max):
    # observed series up to `until`, then monthly climatology (years <= clim_from_year_max)
    ts = ts.asfreq("MS")
    clim = ts[ts.index.year <= clim_from_year_max].groupby(
        ts[ts.index.year <= clim_from_year_max].index.month).mean()
    idx = pd.date_range(ts.index.min(), until, freq="MS")
    out = ts.reindex(idx)
    fut = out.index > ts.index.max()
    out[fut] = out.index[fut].month.map(clim).values
    return out.fillna(pd.Series(out.index.month.map(clim).values, index=out.index))

def well_block(w):
    b = wells.loc[wells["Well No"] == w, "Block / Mandal"]
    return str(b.values[0]) if len(b) else "Depalpur"

RAIN_FEAT = {b: rain_features(_block_series(b)) for b in
             set(well_block(w) for w in well_list_v3)}
print("Rain feature blocks:", {b: f.shape for b, f in RAIN_FEAT.items()})

In [ ]:
# ========== CELL 4 — ZONES, CONDUCTANCE GRAPH, NODE FEATURES (<=2017) =========
def zone_of(w):
    a = aq_info.get(w, {})
    z = a.get("dominant", "Other")
    if z in ZONES:
        return z
    pct = {"Weathered": a.get("wthr_pct", 0), "Fractured": a.get("frac_pct", 0),
           "Massive": a.get("mass_pct", 0)}
    z = max(pct, key=pct.get) if sum(pct.values()) > 0 else "Massive"
    print(f"  {w}: zone 'Other' -> {z}")
    return z

def teff(w):
    a = aq_info.get(w, {})
    p = np.array([a.get("wthr_pct", 0), a.get("mass_pct", 0), a.get("frac_pct", 0)], float)
    if p.sum() <= 0:
        return T_EFF[zone_of(w)]
    return float(p @ np.array([T_EFF["Weathered"], T_EFF["Massive"], T_EFF["Fractured"]]) / p.sum())

meta = wells.drop_duplicates("Well No").set_index("Well No")
elev_all = pd.to_numeric(meta["Elevation of Ground Level"], errors="coerce")
WELLS_ALL = list(well_list_v3)
lat = np.array([float(meta.loc[w, "Northing"]) for w in WELLS_ALL])
lon = np.array([float(meta.loc[w, "Easting"]) for w in WELLS_ALL])
KX, KY = 111.32 * np.cos(np.radians(np.nanmean(lat))), 111.32
D_KM = np.sqrt(((lon[:, None] - lon[None]) * KX) ** 2 + ((lat[:, None] - lat[None]) * KY) ** 2)
T = np.array([teff(w) for w in WELLS_ALL])
T_h = 2 * T[:, None] * T[None] / (T[:, None] + T[None])
W_ADJ = np.where((D_KM > 0) & (D_KM <= GRAPH_RADIUS_KM), T_h / np.maximum(D_KM, 0.1), 0.0)
for i in range(len(WELLS_ALL)):                     # isolated well -> link nearest
    if W_ADJ[i].sum() == 0:
        j = np.argsort(D_KM[i])[1]
        W_ADJ[i, j] = W_ADJ[j, i] = T_h[i, j] / max(D_KM[i, j], 0.1)
W_ADJ = W_ADJ / W_ADJ.max()
A_ = W_ADJ + np.eye(len(WELLS_ALL))
dinv = 1 / np.sqrt(A_.sum(1))
A_HAT = torch.tensor(dinv[:, None] * A_ * dinv[None], dtype=torch.float32, device=DEVICE)
W_ADJ_T = torch.tensor(W_ADJ, dtype=torch.float32, device=DEVICE)

nf = []
for w in WELLS_ALL:
    s = monthly_v3[w]; s = s[s.index.year <= TRAIN_END]
    a = aq_info.get(w, {})
    slope = stats.theilslopes(s.values, np.arange(len(s)) / 12)[0] if len(s) > 12 else 0.0
    e = elev_all.get(w, np.nan)
    nf.append([e if np.isfinite(e) else np.nanmedian(elev_all), s.mean(), s.std(), slope,
               a.get("wthr_pct", 0), a.get("mass_pct", 0), a.get("frac_pct", 0),
               np.log(teff(w))])
nf = np.nan_to_num(np.array(nf, float))
nf = (nf - nf.mean(0)) / (nf.std(0) + 1e-8)
NODE_F = torch.tensor(nf, dtype=torch.float32, device=DEVICE)
ZONE_ID = {w: ZONES.index(zone_of(w)) for w in WELLS_ALL}
print(f"Graph: {len(WELLS_ALL)} nodes, {(W_ADJ > 0).sum() // 2} edges "
      f"(radius {GRAPH_RADIUS_KM} km, weight = T_harm/L)")
print("Zones:", pd.Series({w: ZONES[z] for w, z in ZONE_ID.items()}).value_counts().to_dict())

In [ ]:
# ================= CELL 5 — WINDOWS (identical loop to 04) ====================
H, R, Y, P, WI, Z, DT, SPL = [], [], [], [], [], [], [], []
SC = {}
skipped = []
for w in WELLS_ALL:
    ms = monthly_v3[w]
    trn = ms[ms.index.year <= VAL_END].values.reshape(-1, 1)
    if len(trn) < SEQ_LEN + HORIZON_TRIM:
        skipped.append(w); continue
    sc = MinMaxScaler().fit(trn)                    # same as 04 (<=2019)
    SC[w] = sc
    x = sc.transform(ms.values.reshape(-1, 1)).ravel()
    rf = RAIN_FEAT[well_block(w)].reindex(ms.index).values
    wi = WELLS_ALL.index(w)
    for i in range(len(x) - SEQ_LEN - HORIZON_TRIM + 1):
        t = i + SEQ_LEN
        d = ms.index[t]
        H.append(x[i:t]); R.append(rf[i:t]); Y.append(x[t]); P.append(x[t - 1])
        WI.append(wi); Z.append(ZONE_ID[w]); DT.append(d)
        SPL.append("train" if d.year <= TRAIN_END else "val" if d.year <= VAL_END else "test")

H = torch.tensor(np.array(H), dtype=torch.float32)
R = torch.tensor(np.nan_to_num(np.array(R)), dtype=torch.float32)
Y = torch.tensor(np.array(Y), dtype=torch.float32)
P = torch.tensor(np.array(P), dtype=torch.float32)
WI = torch.tensor(WI); Z = torch.tensor(Z)
DT = pd.DatetimeIndex(DT); SPL = np.array(SPL)
DATE_CODE = torch.tensor(DT.year * 12 + DT.month - 1)

# per-well inverse scaling: metres = (x - min_) / scale_
SC_MIN = torch.zeros(len(WELLS_ALL)); SC_SCALE = torch.ones(len(WELLS_ALL))
for w, sc in SC.items():
    SC_MIN[WELLS_ALL.index(w)] = float(sc.min_[0]); SC_SCALE[WELLS_ALL.index(w)] = float(sc.scale_[0])
to_m = lambda xs, wi: (xs - SC_MIN[wi]) / SC_SCALE[wi]

# storage bounds per zone from observed monthly dh (train)
tr = SPL == "train"
dh_tr = ((Y - P) / SC_SCALE[WI]).numpy()
DH_LO = torch.tensor([np.percentile(dh_tr[tr & (Z.numpy() == k)], 1) for k in range(3)])
DH_HI = torch.tensor([np.percentile(dh_tr[tr & (Z.numpy() == k)], 99) for k in range(3)])

IDX = {s: np.where(SPL == s)[0] for s in ["train", "val", "test"]}
print({s: len(v) for s, v in IDX.items()}, "| skipped:", skipped)
print("Storage bounds dh (m/month):",
      {ZONES[k]: (round(float(DH_LO[k]), 2), round(float(DH_HI[k]), 2)) for k in range(3)})
print("Test samples per well (must equal N in baseline_perwell_metrics.csv):")
print(pd.Series(np.array(WELLS_ALL)[WI.numpy()[IDX['test']]]).value_counts().describe().round(1).to_dict())

SC_SCALE_D, DH_LO_D, DH_HI_D = SC_SCALE.to(DEVICE), DH_LO.to(DEVICE), DH_HI.to(DEVICE)

In [ ]:
# ============================ CELL 6 — MODELS =================================
SPEC = {  #        rain   gcn    zoned  physics  hidden
    "M1":    dict(rain=False, gcn=False, zoned=False, phys=False, hidden=HIDDEN),
    "M2":    dict(rain=True,  gcn=False, zoned=False, phys=False, hidden=HIDDEN),
    "M2big": dict(rain=True,  gcn=False, zoned=False, phys=False, hidden=112),  # ~ M3.5 params
    "M3":    dict(rain=True,  gcn=True,  zoned=False, phys=False, hidden=HIDDEN),
    "M3.5":  dict(rain=True,  gcn=False, zoned=True,  phys=False, hidden=HIDDEN),
    "M3.5G": dict(rain=True,  gcn=True,  zoned=True,  phys=False, hidden=HIDDEN),
    "M4":    dict(rain=True,  gcn=True,  zoned=True,  phys=True,  hidden=HIDDEN),
}

class GCN(nn.Module):
    def __init__(self, fin, fout):
        super().__init__()
        self.l1, self.l2 = nn.Linear(fin, fout), nn.Linear(fout, fout)
    def forward(self, X, A):
        return torch.tanh(self.l2(A @ F.relu(self.l1(A @ X))))

class Branch(nn.Module):
    def __init__(self, fin, hidden):
        super().__init__()
        self.lstm = nn.LSTM(fin, hidden, LAYERS, batch_first=True, dropout=DROPOUT)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        o, _ = self.lstm(x)
        return self.fc(self.drop(o[:, -1])).squeeze(-1)

class Net(nn.Module):
    def __init__(self, rain, gcn, zoned, phys, hidden=64):
        super().__init__()
        self.rain, self.use_gcn, self.zoned = rain, gcn, zoned
        fin = 1 + (7 if rain else 0) + (GCN_DIM if gcn else 0)
        self.branches = nn.ModuleList([Branch(fin, hidden) for _ in range(3 if zoned else 1)])
        if gcn:
            self.gcn = GCN(NODE_F.shape[1], GCN_DIM)
    def forward(self, h, r, wi, z):
        parts = [h.unsqueeze(-1)]
        if self.rain:
            parts.append(r)
        if self.use_gcn:
            e = self.gcn(NODE_F, A_HAT)[wi]
            parts.append(e.unsqueeze(1).expand(-1, h.shape[1], -1))
        x = torch.cat(parts, -1)
        if not self.zoned:
            return self.branches[0](x)
        out = torch.zeros(h.shape[0], device=h.device)
        for k in range(3):
            m = z == k
            if m.any():
                out[m] = self.branches[k](x[m])
        return out

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

for k, v in SPEC.items():
    print(f"{k:6s} params = {n_params(Net(**v)):,}")

In [ ]:
# ======================== CELL 7 — TRAINING ENGINE ============================
def physics_loss(pred, prev, wi, z, dcode):
    dh = (pred - prev) / SC_SCALE_D[wi]                             # metres
    l_st = (F.relu(dh - DH_HI_D[z]) ** 2 + F.relu(DH_LO_D[z] - dh) ** 2).mean()
    num, den = 0.0, 0.0
    for d in torch.unique(dcode):
        m = dcode == d
        if m.sum() < 2:
            continue
        w = W_ADJ_T[wi[m]][:, wi[m]]
        diff = dh[m][:, None] - dh[m][None, :]
        num = num + (w * diff ** 2).sum(); den = den + w.sum()
    l_sp = num / den if float(den) > 0 else torch.zeros((), device=pred.device)
    return l_sp, l_st

def date_batches(idx, rng):
    codes = DATE_CODE[idx].numpy()
    u = np.unique(codes); rng.shuffle(u)
    for k in range(0, len(u), DATES_PER_BATCH):
        yield idx[np.isin(codes, u[k:k + DATES_PER_BATCH])]

def _to(ix):
    ix = torch.as_tensor(ix)
    return (H[ix].to(DEVICE), R[ix].to(DEVICE), Y[ix].to(DEVICE), P[ix].to(DEVICE),
            WI[ix].to(DEVICE), Z[ix].to(DEVICE), DATE_CODE[ix].to(DEVICE))

@torch.no_grad()
def predict(model, idx, bs=4096):
    model.eval(); out = []
    for k in range(0, len(idx), bs):
        h, r, _, _, wi, z, _ = _to(idx[k:k + bs])
        out.append(model(h, r, wi, z).cpu())
    return torch.cat(out)

def rmse_m(model, idx):
    p = predict(model, idx)
    return float(torch.sqrt(((to_m(p, WI[idx]) - to_m(Y[idx], WI[idx])) ** 2).mean()))

def train(name, seed, lam=0.0, train_idx=None, val_idx=None, fixed_epochs=None, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed); rng = np.random.default_rng(seed)
    spec = SPEC[name]
    model = Net(**spec).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=10)
    train_idx = IDX["train"] if train_idx is None else train_idx
    val_idx = IDX["val"] if val_idx is None else val_idx
    best, best_state, best_ep, bad, hist = np.inf, None, 0, 0, []
    n_ep = fixed_epochs or MAX_EPOCHS
    t0 = time.time()
    for ep in range(1, n_ep + 1):
        model.train(); tl = []
        for b in date_batches(train_idx, rng):
            h, r, y, p, wi, z, dc = _to(b)
            pred = model(h, r, wi, z)
            loss = F.mse_loss(pred, y)
            if spec["phys"] and lam > 0:
                l_sp, l_st = physics_loss(pred, p, wi, z, dc)
                loss = loss + lam * (l_sp + l_st)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl.append(loss.item())
        if fixed_epochs:
            hist.append((ep, np.mean(tl), np.nan)); continue
        v = rmse_m(model, val_idx); sch.step(v); hist.append((ep, np.mean(tl), v))
        if v < best - 1e-4:
            best, best_state, best_ep, bad = v, copy.deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
        if bad >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    if verbose:
        print(f"  {name:6s} seed {seed} lam {lam:<6} | best epoch {best_ep or n_ep:3d} | " +
              (f"val RMSE {best:.3f} m" if not fixed_epochs else "full-record refit") +
              f" | {time.time() - t0:.0f}s")
    return model, (best_ep or n_ep), best, hist

In [ ]:
# ====== CELL 8 — CHOOSE λ FOR M4 ON VALIDATION (mean over ALL seeds) =========
# trained M4 models are cached and reused in Cell 9 (no extra training)
lam_rows, LAM_CACHE = [], {}
for lam in LAMBDA_GRID:
    for seed in SEEDS:
        model, ep, v, _ = train("M4", seed, lam)
        LAM_CACHE[(lam, seed)] = (model, ep, v)
        lam_rows.append({"lambda": lam, "seed": seed, "best_epoch": ep, "val_RMSE_m": v})
lam_all = pd.DataFrame(lam_rows)
lam_tab = lam_all.groupby("lambda").val_RMSE_m.agg(["mean", "std"]).reset_index()
LAMBDA = float(lam_tab.loc[lam_tab["mean"].idxmin(), "lambda"])
lam_all.to_csv(os.path.join(OUT_DIR, "M4_lambda_selection.csv"), index=False)
print(lam_tab.round(4).to_string(index=False)); print("Chosen λ =", LAMBDA)

In [ ]:
# ============ CELL 9 — TRAIN ALL MODELS × SEEDS, PREDICT TEST =================
os.makedirs(os.path.join(OUT_DIR, "weights"), exist_ok=True)
pred_rows, run_rows = [], []
te = IDX["test"]
for name in MODELS:
    for seed in SEEDS:
        lam = LAMBDA if SPEC[name]["phys"] else 0.0
        if SPEC[name]["phys"] and (lam, seed) in LAM_CACHE:
            model, ep, v = LAM_CACHE[(lam, seed)]          # reuse from Cell 8
        else:
            model, ep, v, hist = train(name, seed, lam)
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "weights", f"{name}_s{seed}.pt"))
        p = to_m(predict(model, te), WI[te]).numpy()
        pred_rows.append(pd.DataFrame({"Model": name, "Seed": seed,
                                       "Well": np.array(WELLS_ALL)[WI[te].numpy()],
                                       "Date": DT[te], "Obs": to_m(Y[te], WI[te]).numpy(),
                                       "Pred": p}))
        run_rows.append({"Model": name, "Seed": seed, "lambda": lam, "best_epoch": ep,
                         "val_RMSE_m": v, "params": n_params(model)})
preds = pd.concat(pred_rows, ignore_index=True)
runs = pd.DataFrame(run_rows)
preds.to_csv(os.path.join(OUT_DIR, "test_predictions_all.csv"), index=False)
runs.to_csv(os.path.join(OUT_DIR, "runs.csv"), index=False)

In [ ]:
# ===================== CELL 10 — METRICS + ABLATION TABLE =====================
def metrics(obs, pred):
    obs, pred = np.asarray(obs, float), np.asarray(pred, float)
    if len(obs) < 5:
        return None
    sse, sst = np.sum((obs - pred) ** 2), np.sum((obs - obs.mean()) ** 2)
    return {"R2": r2_score(obs, pred) if obs.std() > 0 else 0.0,
            "RMSE": float(np.sqrt(mean_squared_error(obs, pred))),
            "NSE": float(1 - sse / sst) if sst > 0 else np.nan,
            "MAE": float(np.mean(np.abs(obs - pred))),
            "Bias": float(np.mean(pred - obs)), "N": int(len(obs))}

zone_name = {w: ZONES[ZONE_ID[w]] for w in WELLS_ALL}

# (a) per seed -> district mean of per-well metrics -> mean ± SD over seeds
ps = [{"Model": m, "Seed": s, "Well": w, **metrics(g.Obs, g.Pred)}
      for (m, s, w), g in preds.groupby(["Model", "Seed", "Well"]) if metrics(g.Obs, g.Pred)]
ps = pd.DataFrame(ps)
seed_mean = ps.groupby(["Model", "Seed"])[["RMSE", "R2", "Bias"]].mean().reset_index()
abl = seed_mean.groupby("Model").agg(RMSE_mean=("RMSE", "mean"), RMSE_sd=("RMSE", "std"),
                                     R2_mean=("R2", "mean"), R2_sd=("R2", "std"))
abl["params"] = runs.groupby("Model").params.first()
abl["val_RMSE"] = runs.groupby("Model").val_RMSE_m.mean()
abl = abl.reindex(MODELS)

# (b) seed-ensemble prediction -> per-well metrics (files compatible with 04/06/07)
ens = preds.groupby(["Model", "Well", "Date"]).agg(Obs=("Obs", "first"), Pred=("Pred", "mean")).reset_index()
pw = pd.DataFrame([{"Model": m, "Well": w, "Zone": zone_name[w], **metrics(g.Obs, g.Pred)}
                   for (m, w), g in ens.groupby(["Model", "Well"]) if metrics(g.Obs, g.Pred)])
for m in MODELS:
    tag = m.replace(".", "").lower()
    pw[pw.Model == m].drop(columns="Model").round(4).to_csv(
        os.path.join(OUT_DIR, f"eval_{tag}_clean.csv"), index=False)
pw.round(4).to_csv(os.path.join(OUT_DIR, "eval_all_clean.csv"), index=False)

zt = pw.groupby(["Zone", "Model"])[["RMSE", "R2", "Bias"]].mean().unstack("Model")
abl.round(3).to_csv(os.path.join(OUT_DIR, "Table_ablation_clean.csv"))
zt.round(3).to_csv(os.path.join(OUT_DIR, "Table_ablation_by_zone_clean.csv"))
pd.set_option("display.width", 160)
print("ABLATION (test 2020-25; mean of per-well metrics; ± = SD over seeds)")
print(abl.round(3).to_string())
print("\nBy zone (seed-ensemble):"); print(zt.round(3).to_string())

In [ ]:
# ================= CELL 11 — PAIRED TESTS (models and baselines) ==============
def paired(a_name, b_name, tab):
    a = tab[tab.Model == a_name].set_index("Well").RMSE
    b = tab[tab.Model == b_name].set_index("Well").RMSE
    c = a.index.intersection(b.index)
    d = (a[c] - b[c]).values
    rng = np.random.default_rng(0)
    boot = [rng.choice(d, len(d)).mean() for _ in range(10000)]
    p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return {"A": a_name, "B": b_name, "n_wells": len(c), "mean_dRMSE_A_minus_B": d.mean(),
            "CI95_low": np.percentile(boot, 2.5), "CI95_high": np.percentile(boot, 97.5),
            "A_better_in_wells": int((d < 0).sum()), "wilcoxon_p": p}

tab = pw[["Model", "Well", "RMSE"]].copy()
if os.path.exists(BASELINE_CSV):
    bl = pd.read_csv(BASELINE_CSV)
    keep = {"B1_Persistence": "Persistence", "B3_SeasonalPersist": "SeasonalPersist",
            "B4_RidgeARX": "RidgeARX", "B5_ZoneGBM": "ZoneGBM", "B6_PooledGBM": "PooledGBM"}
    bl = bl[bl.Model.isin(keep)].assign(Model=lambda d: d.Model.map(keep))
    chk = pw[pw.Model == "M1"].set_index("Well").N.to_frame("N_dl").join(
        bl[bl.Model == "Persistence"].set_index("Well").N.to_frame("N_bl"), how="inner")
    print("Test-sample count identical to baselines in",
          int((chk.N_dl == chk.N_bl).sum()), "of", len(chk), "wells")
    tab = pd.concat([tab, bl[["Model", "Well", "RMSE"]]], ignore_index=True)
else:
    print("Baseline CSV not found -> only model-vs-model tests")

pairs = [("M2", "M1"),        # rain
         ("M3", "M2"),        # GCN (no zones)
         ("M3.5", "M2"),      # zones
         ("M3.5", "M2big"),   # zones at equal parameter count
         ("M2big", "M2"),     # capacity alone
         ("M3.5G", "M3.5"),   # GCN inside zone model
         ("M4", "M3.5G"),     # physics alone
         ("M4", "M3.5"), ("M3.5", "M1"), ("M4", "M1"), ("M3", "M1")]
pairs += [(m, b) for m in MODELS for b in ["Persistence", "RidgeARX", "ZoneGBM"]
          if b in set(tab.Model)]
pt = pd.DataFrame([paired(a, b, tab) for a, b in pairs])
pt.round(4).to_csv(os.path.join(OUT_DIR, "Table_paired_tests_clean.csv"), index=False)
print(pt.round(3).to_string(index=False))
print("\nNegative mean_dRMSE = A has lower RMSE than B.")

---
## Part B — defensible future projection
1. **Free-run hindcast**: models trained ≤2019 run autoregressively through 2020–2025 with *climatological* rainfall (same conditions as the 2040 run) and are scored against observations, next to two simple benchmarks: (i) 2015–19 monthly climatology, (ii) Theil–Sen trend (≤2019) + monthly climatology.
2. **2040 scenario**: refit on the full record 1998–2025 (fixed epochs = median best epoch from Part A; scaler and node features kept at training-period statistics), climatological rainfall, 3 seeds. Change = mean(2040) − mean(last 12 observed months). Reported with Theil–Sen extrapolation, training-range exceedance, and a reversion diagnostic: ρ(change, last-12-month head − 1998–2019 mean). Strongly negative ρ = projection is mostly reversion to the training mean → present as a stationary-climate/abstraction scenario, not a prediction.

In [ ]:
# ===================== CELL 12 — FREE-RUN HELPERS =============================
def rain_feat_extended(block, obs_until, end, clim_year_max):
    ts = _block_series(block).asfreq("MS")
    ts = ts[ts.index <= obs_until]
    ext = extend_with_climatology(ts, end, clim_year_max)
    return rain_features(ext)

@torch.no_grad()
def free_run(model, w, hist_m, months, rf):
    # hist_m: last SEQ_LEN observed heads (m); months: DatetimeIndex to predict
    model.eval()
    wi = WELLS_ALL.index(w); sc = SC[w]
    h = list(sc.transform(np.asarray(hist_m).reshape(-1, 1)).ravel())
    first = months[0] - pd.DateOffset(months=SEQ_LEN)
    rmat = rf.reindex(pd.date_range(first, months[-1], freq="MS")).values
    wi_t = torch.tensor([wi], device=DEVICE); z_t = torch.tensor([ZONE_ID[w]], device=DEVICE)
    out = []
    for k in range(len(months)):
        ht = torch.tensor([h[-SEQ_LEN:]], dtype=torch.float32, device=DEVICE)
        rt = torch.tensor(np.nan_to_num(rmat[k:k + SEQ_LEN])[None], dtype=torch.float32, device=DEVICE)
        v = float(model(ht, rt, wi_t, z_t)); h.append(v)
        out.append((v - sc.min_[0]) / sc.scale_[0])
    return np.array(out)

def theilsen_clim_forecast(s, months, fit_until):
    s = s[s.index.year <= fit_until]
    ann = s.groupby(s.index.year).mean()
    ann = ann[s.groupby(s.index.year).size() >= 6]
    slope, icpt = stats.theilslopes(ann.values, ann.index.values)[:2]
    resid = s - (icpt + slope * (s.index.year + (s.index.month - 0.5) / 12))
    clim = resid.groupby(resid.index.month).mean()
    t = months.year + (months.month - 0.5) / 12
    return icpt + slope * t + months.month.map(clim).fillna(0).values, slope

In [ ]:
# ============== CELL 13 — FREE-RUN HINDCAST 2020-2025 vs BENCHMARKS ===========
if RUN_PART_B:
    hc_rows = []
    rf_hc = {b: rain_feat_extended(b, f"{VAL_END}-12-01", "2025-12-01", VAL_END) for b in RAIN_FEAT}
    models_s = {(m, s): Net(**SPEC[m]).to(DEVICE) for m in FORECAST_MODELS for s in SEEDS}
    for (m, s), net in models_s.items():
        net.load_state_dict(torch.load(os.path.join(OUT_DIR, "weights", f"{m}_s{s}.pt"),
                                       map_location=DEVICE))
    for w in SC:
        ms = monthly_v3[w]
        pre, post = ms[ms.index.year <= VAL_END], ms[ms.index.year > VAL_END]
        if len(pre) < SEQ_LEN or len(post) < 12:
            continue
        months = pd.date_range(pre.index[-1] + pd.DateOffset(months=1), post.index[-1], freq="MS")
        obs = post.reindex(months)
        ok = obs.notna().values
        last5 = pre[pre.index.year >= VAL_END - 4]
        clim5 = months.month.map(last5.groupby(last5.index.month).mean()).values
        tsf, _ = theilsen_clim_forecast(ms, months, VAL_END)
        rec = {"Well": w, "Zone": zone_name[w], "n": int(ok.sum()),
               "Climatology_RMSE": np.sqrt(np.nanmean((clim5[ok] - obs.values[ok]) ** 2)),
               "TheilSen_RMSE": np.sqrt(np.nanmean((tsf[ok] - obs.values[ok]) ** 2))}
        for m in FORECAST_MODELS:
            runs_m = [free_run(models_s[(m, s)], w, pre.values[-SEQ_LEN:], months,
                               rf_hc[well_block(w)]) for s in SEEDS]
            pm = np.mean(runs_m, 0)
            rec[f"{m}_RMSE"] = np.sqrt(np.mean((pm[ok] - obs.values[ok]) ** 2))
            rec[f"{m}_meanerr"] = float(np.mean(pm[ok] - obs.values[ok]))   # level drift
        hc_rows.append(rec)
    hc = pd.DataFrame(hc_rows)
    hc.round(3).to_csv(os.path.join(OUT_DIR, "hindcast_freerun_2020_2025.csv"), index=False)
    cols = ["Climatology_RMSE", "TheilSen_RMSE"] + [f"{m}_RMSE" for m in FORECAST_MODELS]
    print("FREE-RUN HINDCAST 2020-25 (no observed head after 2019, climatological rain)")
    print("Mean per-well RMSE (m):"); print(hc[cols].mean().round(3).to_string())
    print("\nBy zone:"); print(hc.groupby("Zone")[cols].mean().round(3).to_string())
    best = hc[cols].idxmin(axis=1).value_counts()
    print("\nBest method per well:"); print(best.to_string())
    print("\nMean level error 2020-25 (m; negative = model drifts below observed):")
    print(hc[[f"{m}_meanerr" for m in FORECAST_MODELS]].describe().round(2).to_string())

In [ ]:
# ===== CELL 14 — REFIT ON FULL RECORD (preprocessing fixed) + 2040 SCENARIO =====
if RUN_PART_B:
    all_idx = np.arange(len(Y))
    fc_rows = []
    rf_fc = {b: rain_feat_extended(b, "2025-12-01", FORECAST_END, 2025) for b in RAIN_FEAT}
    for m in FORECAST_MODELS:
        ep = int(runs[runs.Model == m].best_epoch.median())
        lam = LAMBDA if SPEC[m]["phys"] else 0.0
        nets = [train(m, s, lam, train_idx=all_idx, val_idx=IDX["val"],
                      fixed_epochs=ep, verbose=True)[0] for s in SEEDS]
        for w in SC:
            ms = monthly_v3[w]
            months = pd.date_range(ms.index[-1] + pd.DateOffset(months=1), FORECAST_END, freq="MS")
            pm = np.mean([free_run(n, w, ms.values[-SEQ_LEN:], months, rf_fc[well_block(w)])
                          for n in nets], 0)
            f = pd.Series(pm, index=months)
            last12 = ms.iloc[-12:].mean()
            h2040 = f[f.index.year == 2040].mean()
            tsf, slope = theilsen_clim_forecast(ms, months, 2025)
            tr_s = ms[ms.index.year <= VAL_END]
            dev = last12 - tr_s.mean()
            fc_rows.append({"Well": w, "Zone": zone_name[w], "Model": m,
                            "Last_obs": ms.index[-1].strftime("%Y-%m"),
                            "Head_2025_m": round(last12, 2), "Head_2040_m": round(h2040, 2),
                            "Change_m": round(h2040 - last12, 2),
                            "TheilSen_change_m": round(pd.Series(tsf, index=months)[months.year == 2040].mean() - last12, 2),
                            "Hist_slope_m_yr": round(slope, 3),
                            "Dev_from_train_mean_m": round(dev, 2),
                            "Outside_training_range": bool(h2040 > tr_s.max() or h2040 < tr_s.min())})
    fc = pd.DataFrame(fc_rows)
    for m in FORECAST_MODELS:
        fc[fc.Model == m].to_csv(os.path.join(OUT_DIR, f"fc_{m.replace('.', '').lower()}_clean.csv"), index=False)
    fc.to_csv(os.path.join(OUT_DIR, "fc_all_clean.csv"), index=False)
    s = fc.groupby("Model").agg(mean_change=("Change_m", "mean"), sd_change=("Change_m", "std"),
                                n_decline_5m=("Change_m", lambda x: int((x <= -5).sum())),
                                n_outside_range=("Outside_training_range", "sum"),
                                rho_vs_TheilSen=("Change_m", lambda x: stats.spearmanr(
                                    x, fc.loc[x.index, "TheilSen_change_m"]).correlation),
                                rho_vs_dev_from_mean=("Change_m", lambda x: stats.spearmanr(
                                    x, fc.loc[x.index, "Dev_from_train_mean_m"]).correlation),
                                rising_hist_but_decline=("Change_m", lambda x: int(
                                    ((fc.loc[x.index, "Hist_slope_m_yr"] > 0) & (x < 0)).sum())))
    print("2040 SCENARIO SUMMARY"); print(s.reindex(FORECAST_MODELS).round(2).to_string())
    print("\nrho_vs_dev_from_mean < -0.6  -> projection is mainly reversion to the 1998-2019 mean")

In [ ]:
# ======================= CELL 15 — SAVE RUN RECORD (FAIR) =====================
record = {"seq_len": SEQ_LEN, "train_end": TRAIN_END, "val_end": VAL_END, "seeds": SEEDS,
          "hidden": {m: SPEC[m]["hidden"] for m in MODELS}, "models": MODELS, "layers": LAYERS, "dropout": DROPOUT, "gcn_dim": GCN_DIM,
          "lr": LR, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
          "lambda_grid": LAMBDA_GRID, "lambda_chosen": LAMBDA,
          "graph_radius_km": GRAPH_RADIUS_KM, "T_eff": T_EFF,
          "n_wells": len(SC), "n_train": len(IDX["train"]), "n_val": len(IDX["val"]),
          "n_test": len(IDX["test"]), "torch": torch.__version__, "device": str(DEVICE)}
json.dump(record, open(os.path.join(OUT_DIR, "run_record.json"), "w"), indent=2)
print(json.dumps(record, indent=2))
print("\nSend me the printed outputs of Cells 8, 10, 11, 13, 14 (or export the notebook as PDF again)")